# Punto 6: Base de Conocimiento de Productos y Fundación RAG

**Objetivo:** Construir una base de conocimiento simulada (JSON/diccionario Python) con descripciones atractivas para 3 de los productos más vendidos, incluyendo sus precios promedio reales extraídos del dataset.

## Enfoque

1. **Seleccionar 3 productos** de los más vendidos identificados en el Punto 1 — uno de cada una de las 3 categorías con mayor ingreso
2. **Extraer métricas reales** del dataset: precio promedio, unidades vendidas, ingresos, puntuación de reseñas
3. **Escribir descripciones atractivas** que potencien un Agente de IA conversacional (contexto RAG)
4. **Estructurar la base de conocimiento** en un formato listo para embedding y retrieval

**Criterio de Selección de Productos:**
- Producto más vendido de cada una de las **3 categorías de mayor ingreso** (Health & Beauty, Watches & Gifts, Bed Bath & Table)
- Todos los precios y métricas se extraen directamente del dataset — sin números fabricados

---
## 1. Extracción de Datos — Métricas Reales del Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import warnings

warnings.filterwarnings('ignore')

COLORS = {
    'primary': '#0033a0', 'secondary': '#6699cc', 'accent': '#e3e82a',
    'danger': '#c73e1d', 'neutral': '#001a4d', 'inactive': '#d0d0d0'
}

DATA_PATH = '../Dataset_prueba/'

orders = pd.read_csv(f'{DATA_PATH}orders_dataset.csv')
order_items = pd.read_csv(f'{DATA_PATH}order_items_dataset.csv')
products = pd.read_csv(f'{DATA_PATH}products_dataset.csv')
categories = pd.read_csv(f'{DATA_PATH}product_category_name_translation.csv')
reviews = pd.read_csv(f'{DATA_PATH}order_reviews_dataset.csv')

delivered = orders[orders['order_status'] == 'delivered']
sales = (
    order_items.merge(delivered[['order_id']], on='order_id')
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(categories, on='product_category_name', how='left')
)
sales['category'] = sales['product_category_name_english'].fillna('uncategorized')

# Reseñas por producto
# Deduplicar: conservar solo el ítem más caro por pedido para que una reseña
# no se cuente una vez por ítem en pedidos multi-producto (evita inflación).
prod_reviews = reviews.merge(order_items[['order_id', 'product_id', 'price']], on='order_id')
prod_reviews = prod_reviews.sort_values('price', ascending=False).drop_duplicates(
    subset=['review_id', 'order_id'], keep='first'
)
prod_reviews = prod_reviews.drop(columns=['price'])
review_agg = prod_reviews.groupby('product_id').agg(
    avg_review=('review_score', 'mean'),
    review_count=('review_score', 'count'),
    pct_5star=('review_score', lambda x: (x == 5).mean() * 100),
    pct_positive=('review_score', lambda x: (x >= 4).mean() * 100)
).round(2)

# Estadísticas por producto
prod_stats = sales.groupby('product_id').agg(
    category=('category', 'first'),
    units_sold=('order_item_id', 'count'),
    total_revenue=('price', 'sum'),
    avg_price=('price', 'mean'),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    avg_freight=('freight_value', 'mean')
).join(review_agg).sort_values('total_revenue', ascending=False)

# Calcular las 3 categorías con mayor ingreso desde los datos
top3_revenue_cats = (
    sales.groupby('category')['price'].sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

# Seleccionar el producto top de cada una de las 3 categorías principales
TARGET_CATEGORIES = ['health_beauty', 'watches_gifts', 'bed_bath_table']

# Validar que TARGET_CATEGORIES coincide con el top 3 real de los datos
assert set(TARGET_CATEGORIES) == set(top3_revenue_cats), (
    f"TARGET_CATEGORIES no coincide! Hardcoded: {TARGET_CATEGORIES}, "
    f"Top 3 real por ingresos: {top3_revenue_cats}"
)
print(f'Validado: TARGET_CATEGORIES {TARGET_CATEGORIES} coincide con top 3 real por ingresos.')

selected_products = {}

for cat in TARGET_CATEGORIES:
    top = prod_stats[prod_stats['category'] == cat].head(1)
    pid = top.index[0]
    selected_products[cat] = {
        'product_id': pid,
        **top.iloc[0].to_dict()
    }

print('\nProductos seleccionados (datos reales del dataset)')
print('=' * 90)
for cat, info in selected_products.items():
    print(f'\n  Categoría: {cat}')
    print(f'  Product ID: {info["product_id"]}')
    print(f'  Unidades vendidas: {info["units_sold"]} | Ingresos: R$ {info["total_revenue"]:,.2f}')
    print(f'  Precio promedio: R$ {info["avg_price"]:.2f} | Rango: R$ {info["min_price"]:.2f} - R$ {info["max_price"]:.2f}')
    print(f'  Reseña promedio: {info["avg_review"]:.2f}/5.0 ({info["review_count"]:.0f} reseñas, {info["pct_positive"]:.0f}% positivas)')


Validado: TARGET_CATEGORIES ['health_beauty', 'watches_gifts', 'bed_bath_table'] coincide con top 3 real por ingresos.

Productos seleccionados (datos reales del dataset)

  Categoría: health_beauty
  Product ID: bb50f2e236e5eea0100680137654686c
  Unidades vendidas: 194 | Ingresos: R$ 63,560.00
  Precio promedio: R$ 327.63 | Rango: R$ 325.00 - R$ 330.00
  Reseña promedio: 4.21/5.0 (188 reseñas, 81% positivas)

  Categoría: watches_gifts
  Product ID: 53b36df67ebb7c41585e8d54d6772e08
  Unidades vendidas: 321 | Ingresos: R$ 37,454.63
  Precio promedio: R$ 116.68 | Rango: R$ 99.90 - R$ 179.90
  Reseña promedio: 4.25/5.0 (300 reseñas, 82% positivas)

  Categoría: bed_bath_table
  Product ID: 99a4788cb24856965c36a24e339b6058
  Unidades vendidas: 477 | Ingresos: R$ 42,049.66
  Precio promedio: R$ 88.15 | Rango: R$ 74.00 - R$ 89.90
  Reseña promedio: 3.95/5.0 (430 reseñas, 71% positivas)


---
## 2. Construcción de la Base de Conocimiento

Construimos la base de conocimiento con **3 capas de información** por producto:
- **Orientada al cliente**: Descripciones atractivas que un Agente de IA usaría en conversaciones
- **Metadata**: Atributos estructurados para filtrado y retrieval
- **Inteligencia de negocio**: Métricas internas para el motor de recomendación

In [2]:
# --- Construcción de la Base de Conocimiento ---
hb = selected_products['health_beauty']
wg = selected_products['watches_gifts']
bbt = selected_products['bed_bath_table']

knowledge_base = {
    "metadata": {
        "version": "1.0",
        "created_from": "Brazilian E-Commerce Dataset (2016-2018)",
        "purpose": "Product knowledge base for AI Agent (RAG retrieval context)",
        "total_products": 3,
        "data_source": "Real metrics extracted from delivered orders dataset",
        "segment_taxonomy": [
            "Loyal Champions", "Premium Buyers", "Mid-Value Buyers", "Low-Engagement Buyers"
        ],
        "rag_ready_fields": ["name", "category", "description", "highlights", "tags"]
    },
    "products": [
        {
            "id": hb['product_id'],
            "name": "Premium Health & Beauty Essential Kit",
            "category": "Health & Beauty",
            "description": (
                f"Our #1 revenue-generating product in the Health & Beauty category — "
                f"the highest-grossing individual product across the entire catalog with "
                f"R$ {hb['total_revenue']:,.0f} in total sales. Purchased by {int(hb['units_sold'])} "
                f"verified buyers with an average rating of {hb['avg_review']:.1f}/5.0 "
                f"({hb['pct_positive']:.0f}% positive reviews). Priced at R$ {hb['avg_price']:.2f}, "
                f"it sits at the premium tier of the category — a choice for customers "
                f"who prioritize quality in their personal care routine."
            ),
            "highlights": [
                f"#1 best-seller by revenue in Health & Beauty (R$ {hb['total_revenue']:,.0f} total)",
                f"{int(hb['units_sold'])} verified purchases with {hb['avg_review']:.1f}/5.0 average rating",
                f"{hb['pct_positive']:.0f}% positive reviews ({hb['pct_5star']:.0f}% rated 5 stars)",
                f"Price range: R$ {hb['min_price']:.2f} – R$ {hb['max_price']:.2f}"
            ],
            "pricing": {
                "average_price": round(hb['avg_price'], 2),
                "price_range": {
                    "min": round(hb['min_price'], 2),
                    "max": round(hb['max_price'], 2)
                },
                "currency": "BRL",
                "average_freight": round(hb['avg_freight'], 2)
            },
            "customer_reviews": {
                "average_score": round(hb['avg_review'], 2),
                "total_reviews": int(hb['review_count']),
                "pct_positive": round(hb['pct_positive'], 1),
                "pct_5_star": round(hb['pct_5star'], 1)
            },
            "sales_metrics": {
                "units_sold": int(hb['units_sold']),
                "total_revenue": round(hb['total_revenue'], 2),
                "category_rank": 1
            },
            "tags": ["best-seller", "premium", "health", "beauty", "skincare", "self-care", "gift-idea"],
            "target_segments": ["Loyal Champions", "Premium Buyers"]
        },
        {
            "id": wg['product_id'],
            "name": "Signature Watch & Gift Collection Piece",
            "category": "Watches & Gifts",
            "description": (
                f"The top-selling product in our Watches & Gifts collection — the #2 revenue "
                f"category on the platform. With {int(wg['units_sold'])} units sold and "
                f"R$ {wg['total_revenue']:,.0f} in total revenue, this piece has become the "
                f"go-to choice for customers seeking gifts and accessories. Rated "
                f"{wg['avg_review']:.1f}/5.0 across {int(wg['review_count'])} reviews "
                f"({wg['pct_positive']:.0f}% positive). At an average price of "
                f"R$ {wg['avg_price']:.2f}, it offers strong perceived value in the "
                f"gifting segment — accessible enough for a wide audience while maintaining "
                f"the premium feel customers expect."
            ),
            "highlights": [
                f"Top-selling product in Watches & Gifts (R$ {wg['total_revenue']:,.0f} revenue)",
                f"{int(wg['units_sold'])} units sold — highest volume in the category",
                f"{wg['avg_review']:.1f}/5.0 average rating ({wg['pct_positive']:.0f}% positive)",
                f"Average price R$ {wg['avg_price']:.2f} — strong value in the gift segment"
            ],
            "pricing": {
                "average_price": round(wg['avg_price'], 2),
                "price_range": {
                    "min": round(wg['min_price'], 2),
                    "max": round(wg['max_price'], 2)
                },
                "currency": "BRL",
                "average_freight": round(wg['avg_freight'], 2)
            },
            "customer_reviews": {
                "average_score": round(wg['avg_review'], 2),
                "total_reviews": int(wg['review_count']),
                "pct_positive": round(wg['pct_positive'], 1),
                "pct_5_star": round(wg['pct_5star'], 1)
            },
            "sales_metrics": {
                "units_sold": int(wg['units_sold']),
                "total_revenue": round(wg['total_revenue'], 2),
                "category_rank": 1
            },
            "tags": ["best-seller", "watch", "gift", "accessory", "celebration", "affordable-luxury"],
            "target_segments": ["Loyal Champions", "Premium Buyers", "Mid-Value Buyers"]
        },
        {
            "id": bbt['product_id'],
            "name": "Luxury Bed & Bath Textile Set",
            "category": "Bed, Bath & Table",
            "description": (
                f"The volume champion of our #1 category by units sold — Bed, Bath & Table. "
                f"With {int(bbt['units_sold'])} units sold, this product leads the largest "
                f"category in the catalog by a significant margin. It has generated "
                f"R$ {bbt['total_revenue']:,.0f} in revenue with an average rating of "
                f"{bbt['avg_review']:.1f}/5.0 across {int(bbt['review_count'])} reviews. "
                f"At R$ {bbt['avg_price']:.2f} average price, it represents our most "
                f"accessible entry point among the top sellers — ideal for first-time "
                f"buyers and customers seeking quality home essentials at a practical price."
            ),
            "highlights": [
                f"#1 by volume in the largest product category ({int(bbt['units_sold'])} units sold)",
                f"R$ {bbt['total_revenue']:,.0f} total revenue — consistent demand",
                f"{bbt['avg_review']:.1f}/5.0 average rating ({bbt['pct_positive']:.0f}% positive)",
                f"Most accessible top seller at R$ {bbt['avg_price']:.2f} average price"
            ],
            "pricing": {
                "average_price": round(bbt['avg_price'], 2),
                "price_range": {
                    "min": round(bbt['min_price'], 2),
                    "max": round(bbt['max_price'], 2)
                },
                "currency": "BRL",
                "average_freight": round(bbt['avg_freight'], 2)
            },
            "customer_reviews": {
                "average_score": round(bbt['avg_review'], 2),
                "total_reviews": int(bbt['review_count']),
                "pct_positive": round(bbt['pct_positive'], 1),
                "pct_5_star": round(bbt['pct_5star'], 1)
            },
            "sales_metrics": {
                "units_sold": int(bbt['units_sold']),
                "total_revenue": round(bbt['total_revenue'], 2),
                "category_rank": 1
            },
            "tags": ["best-seller", "home", "bath", "bed", "textiles", "comfort", "affordable"],
            "target_segments": ["Loyal Champions", "Premium Buyers", "Mid-Value Buyers", "Low-Engagement Buyers"]
        }
    ]
}

# Añadir texto de retrieval consolidado por producto (RAG-ready)
for product in knowledge_base['products']:
    highlight_text = ' '.join(product.get('highlights', []))
    tag_text = ' '.join(product.get('tags', []))
    product['retrieval_text'] = (
        f"{product['name']}. Category: {product['category']}. "
        f"{product['description']} Highlights: {highlight_text}. Tags: {tag_text}."
    )

# Validaciones de integridad
product_ids = [p['id'] for p in knowledge_base['products']]
assert len(product_ids) == len(set(product_ids)), 'IDs de producto duplicados en la KB.'
for p in knowledge_base['products']:
    assert p['pricing']['average_price'] > 0, f"Precio promedio inválido para {p['id']}"

print('Base de Conocimiento construida exitosamente!')
print(f'Productos: {len(knowledge_base["products"])}')
for p in knowledge_base['products']:
    print(f'  - {p["name"]} ({p["category"]}) — R$ {p["pricing"]["average_price"]:.2f} precio promedio')


Base de Conocimiento construida exitosamente!
Productos: 3
  - Premium Health & Beauty Essential Kit (Health & Beauty) — R$ 327.63 precio promedio
  - Signature Watch & Gift Collection Piece (Watches & Gifts) — R$ 116.68 precio promedio
  - Luxury Bed & Bath Textile Set (Bed, Bath & Table) — R$ 88.15 precio promedio


In [3]:
# --- Guardar en archivo JSON ---
kb_path = '../data/knowledge_base.json'
import os
os.makedirs('../data', exist_ok=True)

with open(kb_path, 'w', encoding='utf-8') as f:
    json.dump(knowledge_base, f, indent=2, ensure_ascii=False)

print(f'Base de conocimiento guardada en: {kb_path}')
print(f'Tamaño del archivo: {os.path.getsize(kb_path):,} bytes')


Base de conocimiento guardada en: ../data/knowledge_base.json
Tamaño del archivo: 7,971 bytes


---
## 3. Justificación del Diseño de la Base de Conocimiento

### Decisiones de Estructura

La base de conocimiento está diseñada para servir a **múltiples consumidores**:

| Capa | Campos | Consumidor |
|---|---|---|
| **Orientada al cliente** | `name`, `description`, `highlights` | Agente de IA / chatbot para respuestas en lenguaje natural |
| **Metadata estructurada** | `pricing`, `tags`, `category` | Motor de búsqueda/filtrado, sistema de recomendación |
| **Inteligencia de negocio** | `sales_metrics`, `customer_reviews`, `target_segments` | Analítica interna, ranking por segmento |

### ¿Por Qué Estos 3 Productos?

| Producto | Categoría | Justificación de Selección |
|---|---|---|
| Premium Health & Beauty Kit | Health & Beauty (#1 ingresos) | Mayor ingreso individual de todo el dataset |
| Signature Watch Collection | Watches & Gifts (#2 ingresos) | Mayor volumen en categoría premium/regalos — alto potencial de cross-sell |
| Luxury Bed & Bath Set | Bed, Bath & Table (#1 volumen) | Campeón de volumen — mayor alcance de clientes |

### Diseño RAG-Ready

Cada entrada de producto contiene:
- **Campos de texto enriquecido** (`description`, `highlights`) más `retrieval_text` consolidado optimizado para embedding y búsqueda semántica
- **Tags** para retrieval basado en palabras clave como respaldo
- **Segmentos objetivo** vinculados a la segmentación del Punto 3 — el Agente de IA puede priorizar productos según a quién le habla
- **Precios reales** extraídos del dataset — el agente nunca alucinará precios

### Conexión con la Evaluación

Esta base de conocimiento es el **fundamento del Punto 1 de las responsabilidades del cargo (Ecosistemas GenAI / RAG)**:
- En producción, este JSON se embedería en un vector store (ej: Pinecone, Weaviate, Chroma)
- El Agente de IA recupera contexto de producto relevante antes de generar respuestas
- El campo `target_segments` usa la taxonomía actual de segmentos para personalizar el pitch por perfil
- Las `sales_metrics` proporcionan datos de grounding que previenen la alucinación

### Nota sobre Descripciones

Todas las descripciones y highlights se generan dinámicamente con f-strings a partir de métricas reales del dataset (unidades vendidas, ingresos, puntuación de reseñas, precios). Los nombres de producto son simulados ya que el dataset original no contiene nombres legibles — solo IDs de producto y categorías.